# Reto 1 - Caracterización de variables aleatorias

## Caso de estudio

FitBalance Gym es una cadena ficticia de gimnasios que busca analizar el riesgo económico relacionado con la cancelación de membresías.

Frecuencia: indica el número de clientes que cancelan su membresía diariamente (clientes/día).

Severidad: representa el ingreso promedio que el gimnasio deja de recibir por cada cliente que cancela (COP/cliente).

Distribución de pérdidas: resulta de multiplicar la frecuencia por la severidad y permite estimar la pérdida económica diaria causada por las cancelaciones.

## 1. Librerías


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path = "/content/drive/MyDrive/Integracion de datos/Reto_1_Clientes_Gimnasio.xlsx"

In [ ]:
df = pd.read_excel(path, sheet_name='Datos')
df.head()

Freq = pd.to_numeric(df.iloc[:,5], errors='coerce')
Sev = pd.to_numeric(df.iloc[:,6], errors='coerce')
LDA = Freq * Sev

plt.figure()
sns.histplot(LDA, color='blue', bins=10, kde=True)
plt.title('Distribución de pérdidas por cancelación de membresías')
plt.xlabel('Pérdidas')
plt.ylabel('Frecuencia')
plt.grid()
plt.show()

In [ ]:
df = pd.read_excel(path, sheet_name='Datos')
df.head()

Freq = pd.to_numeric(df.iloc[:,5], errors='coerce')
Sev = pd.to_numeric(df.iloc[:,6], errors='coerce')
LDA = Freq * Sev

plt.figure()
sns.histplot(LDA, color='blue', bins=10, kde=True)
plt.title('Distribución de pérdidas por cancelación de membresías')
plt.xlabel('Pérdidas')
plt.ylabel('Frecuencia')
plt.grid()
plt.show()

Se procede con la construcción del histograma de frecuencias

In [ ]:
NI = 10 # Número de intervalos
counts, Limits = np.histogram(LDA, bins=NI) # Agrupar los datos
LI = Limits[:-1] # Límite inferior
LS = Limits[1:] # Límite superior
fi = counts / sum(counts) # Porcentaje de datos por intervalo
MC = (LI + LS) / 2 # Marca de clase

histogram_data = pd.DataFrame({
    'LI': LI,
    'LS': LS,
    'ND': counts,
    'fi': fi,
    'MC': MC
})
display(histogram_data)

Medidas de tendencia central y dispersión

In [ ]:
np.set_printoptions(suppress=True)

# Media
U = np.sum(MC * fi)
print(f"La media de las pérdidas es: {U:.2f}")

# Varianza
varc = np.sum((MC - U)**2 * fi)
print(f"La varianza es: {varc:.2f}")

# Intervalo de variación de las pérdidas
sigma = np.sqrt(varc)
print(f"El límite inferior es: {U - 2*sigma:.2f}")
print(f"El límite superior es: {U + 2*sigma:.2f}")

# Coeficiente de asimetría
cas = np.sum(((MC - U)**3) * fi) / (sigma**3)
print(f"El coeficiente de asimetría es: {cas:.2f}")

# Coeficiente de curtosis
cur = np.sum(((MC - U)**4) * fi) / (sigma**4)
print(f"El coeficiente de curtosis es: {cur:.2f}")

## 5. Medidas solicitadas

In [ ]:
media = np.sum(MC * fi)
varianza = np.sum((MC - media)**2 * fi)
desviacion = np.sqrt(varianza)
asimetria = np.sum((MC - media)**3 * fi) / desviacion**3
curtosis = np.sum((MC - media)**4 * fi) / desviacion**4

resultados = pd.DataFrame({
    "Medida": ["Media", "Varianza", "Desviación estándar", "Asimetría", "Curtosis"],
    "Resultado": [media, varianza, desviacion, asimetria, curtosis]
})

resultados

In [ ]:
def MedidasDTC(Xi, NI=10):
    """Calcula las medidas pedidas por el reto usando el método de datos agrupados visto en clase."""
    Xi = pd.Series(Xi).dropna().to_numpy(dtype=float)

    counts, Limits = np.histogram(Xi, bins=NI)
    LI = Limits[:-1]
    LS = Limits[1:]
    fi = counts / counts.sum()
    MC = (LI + LS) / 2

    media = np.sum(MC * fi)
    varianza = np.sum((MC - media)**2 * fi)
    desviacion = np.sqrt(varianza)
    asimetria = np.sum((MC - media)**3 * fi) / desviacion**3
    curtosis = np.sum((MC - media)**4 * fi) / desviacion**4

    tabla_hist = pd.DataFrame({
        'Limite Inferior': LI,
        'Limite Superior': LS,
        'ND': counts,
        'fi': fi,
        'MC': MC
    })

    medidas = pd.Series({
        'Media': media,
        'Varianza': varianza,
        'Desviación estándar': desviacion,
        'Asimetría': asimetria,
        'Curtosis': curtosis
    })

    return tabla_hist, medidas


## 6. Reto 2 - Muestreo por Montecarlo

De acuerdo con el enunciado, se realiza un muestreo de la **Distribución Agregada de Pérdidas (LDA)** mediante Montecarlo y se comparan las medidas de tendencia central y heterogeneidad **antes y después** del muestreo.

Se trabajarán dos tamaños de simulación:
- **99.9%:** 1.000 simulaciones.
- **99.99%:** 10.000 simulaciones.

> El análisis e interpretación de los resultados se deja para la entrega escrita/oral.


In [ ]:
# Limpiar la distribución original
LDA_original = pd.Series(LDA).dropna().to_numpy(dtype=float)

# Medidas antes del muestreo
hist_original, medidas_original = MedidasDTC(LDA_original, NI=10)

print("Medidas ANTES del muestreo")
display(medidas_original.to_frame("Original"))


### Construcción de la distribución empírica para Montecarlo

Se utilizan los mismos intervalos del histograma de la LDA original. La frecuencia relativa `fi` funciona como probabilidad de seleccionar cada intervalo. Una vez seleccionado el intervalo, se genera aleatoriamente un valor dentro de sus límites.


In [ ]:
# Histograma base de la LDA observada
NI = 10
counts, Limits = np.histogram(LDA_original, bins=NI)
LI = Limits[:-1]
LS = Limits[1:]
probabilidades = counts / counts.sum()

base_montecarlo = pd.DataFrame({
    'LI': LI,
    'LS': LS,
    'ND': counts,
    'Probabilidad': probabilidades
})

display(base_montecarlo)


In [ ]:
def montecarlo_desde_histograma(LI, LS, probabilidades, n, semilla=123):
    """Genera n pérdidas simuladas respetando la distribución empírica del histograma."""
    rng = np.random.default_rng(semilla)

    # 1. Escoger el intervalo según su frecuencia relativa
    intervalos = rng.choice(len(probabilidades), size=n, p=probabilidades)

    # 2. Generar un valor uniforme dentro del intervalo seleccionado
    u = rng.random(n)
    muestra = LI[intervalos] + u * (LS[intervalos] - LI[intervalos])

    return muestra


### Muestreo con confiabilidad 99.9% - 1.000 simulaciones


In [ ]:
LDA_999 = montecarlo_desde_histograma(
    LI, LS, probabilidades,
    n=1000,
    semilla=123
)

hist_999, medidas_999 = MedidasDTC(LDA_999, NI=10)

print("Medidas DESPUÉS del muestreo - 99.9% (1.000 simulaciones)")
display(medidas_999.to_frame("Montecarlo 99.9%"))


### Muestreo con confiabilidad 99.99% - 10.000 simulaciones


In [ ]:
LDA_9999 = montecarlo_desde_histograma(
    LI, LS, probabilidades,
    n=10000,
    semilla=456
)

hist_9999, medidas_9999 = MedidasDTC(LDA_9999, NI=10)

print("Medidas DESPUÉS del muestreo - 99.99% (10.000 simulaciones)")
display(medidas_9999.to_frame("Montecarlo 99.99%"))


## 7. Comparación antes y después


In [ ]:
comparacion = pd.concat([
    medidas_original.rename('Original'),
    medidas_999.rename('Montecarlo 99.9%'),
    medidas_9999.rename('Montecarlo 99.99%')
], axis=1)

# Diferencias porcentuales frente a la distribución original
comparacion['Error % 99.9%'] = ((comparacion['Montecarlo 99.9%'] - comparacion['Original']) / comparacion['Original']) * 100
comparacion['Error % 99.99%'] = ((comparacion['Montecarlo 99.99%'] - comparacion['Original']) / comparacion['Original']) * 100

display(comparacion)


## 8. Visualización de las distribuciones


In [ ]:
plt.figure(figsize=(8,5))
plt.hist(LDA_original, bins=10, density=True, alpha=0.5, label='Original')
plt.hist(LDA_999, bins=10, density=True, alpha=0.5, label='Montecarlo 99.9%')
plt.title('LDA original vs. Montecarlo 99.9%')
plt.xlabel('Pérdida económica diaria (COP)')
plt.ylabel('Densidad')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8,5))
plt.hist(LDA_original, bins=10, density=True, alpha=0.5, label='Original')
plt.hist(LDA_9999, bins=10, density=True, alpha=0.5, label='Montecarlo 99.99%')
plt.title('LDA original vs. Montecarlo 99.99%')
plt.xlabel('Pérdida económica diaria (COP)')
plt.ylabel('Densidad')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 9. Percentiles de pérdida (apoyo para el análisis)


In [ ]:
percentiles = pd.DataFrame({
    'Distribución': ['Original', 'Montecarlo 99.9%', 'Montecarlo 99.99%'],
    'P50': [np.quantile(LDA_original, 0.50), np.quantile(LDA_999, 0.50), np.quantile(LDA_9999, 0.50)],
    'P95': [np.quantile(LDA_original, 0.95), np.quantile(LDA_999, 0.95), np.quantile(LDA_9999, 0.95)],
    'P99': [np.quantile(LDA_original, 0.99), np.quantile(LDA_999, 0.99), np.quantile(LDA_9999, 0.99)],
    'P99.9': [np.quantile(LDA_original, 0.999), np.quantile(LDA_999, 0.999), np.quantile(LDA_9999, 0.999)]
})

display(percentiles)
